# Evaluación Final Transversal - BIY7121 Minería de Datos
## Predicción de lluvia al día siguiente con `weatherAUS.csv`

**Asignatura:** BIY7121 - Minería de Datos  
**Sección:** BIY7121-004D  
**Integrantes:** Diego Casanova / José Soto  
**RUT:** completar antes de subir a AVA  
**Metodología usada:** CRISP-DM  

Este notebook desarrolla el ciclo completo solicitado para el examen transversal: introducción, metodología, preparación de datos, análisis exploratorio, aplicación de técnicas de minería de datos, modelos predictivos, evaluación, descubrimiento de patrones, conclusiones, recomendaciones y propuesta de automatización.

> Nota de integridad: no se inventan datos personales. El campo RUT queda como pendiente porque no venía informado en los archivos de trabajo.

## 0. Cómo ejecutar este notebook

1. Subir el archivo **`weatherAUS.csv`** al entorno de Google Colab o dejarlo en la misma carpeta del notebook.
2. Ejecutar las celdas en orden.
3. El notebook generará tablas, gráficos, métricas, un archivo preparado, un modelo guardado y archivos de salida para respaldo.

El código considera problemas comunes de calidad: columnas con espacios, valores vacíos, fechas en texto, datos faltantes, variables categóricas, posible desbalance de clases y fuga de información.

# 1. Introducción

El objetivo del proyecto es aplicar minería de datos sobre el dataset `weatherAUS.csv` para responder una pregunta concreta: **¿se puede anticipar si lloverá al día siguiente?**

La variable objetivo es `RainTomorrow`, por lo que el problema se trabaja como una **clasificación binaria**. La minería de datos puede apoyar decisiones reales porque permite transformar registros históricos en señales de riesgo, por ejemplo para planificar rutas, turnos, actividades al aire libre, mantenciones o medidas preventivas.

El trabajo no busca reemplazar un servicio meteorológico profesional. Su propósito es construir una solución analítica clara, reproducible y evaluada con métricas, utilizando un enfoque estructurado de minería de datos.

# 2. Metodología de trabajo

Se utiliza **CRISP-DM**, porque ordena el proyecto desde el entendimiento del problema hasta la evaluación y la posibilidad de automatizar la solución.

Las fases aplicadas son:

1. **Comprensión del negocio:** definir el problema y el valor esperado.
2. **Comprensión de datos:** revisar estructura, tipos, nulos, duplicados, distribución y relaciones.
3. **Preparación:** limpiar, transformar, imputar y preparar variables para modelamiento.
4. **Modelamiento:** aplicar técnicas supervisadas y no supervisadas.
5. **Evaluación:** comparar modelos con métricas adecuadas al problema.
6. **Despliegue propuesto:** documentar cómo automatizar el proceso en una operación real.

La técnica principal es clasificación, porque se predice `RainTomorrow`. Además, se usa clustering para descubrir perfiles de días meteorológicos sin utilizar la etiqueta como entrada.

In [1]:
# ============================================================
# 3. Configuración inicial
# ============================================================
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)
plt.rcParams['figure.dpi'] = 120

OUTPUT_DIR = Path('salidas_weatherAUS')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Entorno listo. Carpeta de salida:', OUTPUT_DIR.resolve())

Entorno listo. Carpeta de salida: /content/salidas_weatherAUS


In [4]:
# ============================================================
# 4. Carga del dataset
# ============================================================
import pandas as pd
from pathlib import Path

# Busca primero el archivo local. Si no existe, lo descarga desde GitHub.
archivo_local = Path("weatherAUS.csv")
url_github = (
    "https://raw.githubusercontent.com/"
    "DIECASZU/BIY7121-weatherAUS/main/weatherAUS.csv"
)

fuente = archivo_local if archivo_local.exists() else url_github

df = pd.read_csv(
    fuente,
    na_values=["", " ", "NA", "N/A", "null", "None"],
    low_memory=False
)

# Elimina espacios accidentales en los nombres de las columnas
df.columns = df.columns.str.strip()

print(f"Fuente utilizada: {fuente}")
print(f"Registros: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
df.head()

Fuente utilizada: https://raw.githubusercontent.com/DIECASZU/BIY7121-weatherAUS/main/weatherAUS.csv
Registros: 142,193
Columnas: 24


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RISK_MM,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,WNW,20.0,24.0,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,0.0,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,WSW,4.0,22.0,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,0.0,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,WSW,19.0,26.0,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,0.0,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,E,11.0,9.0,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,1.0,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,NW,7.0,20.0,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,0.2,No


# 3. Preparación y comprensión inicial de los datos

Antes de modelar se revisa la calidad del dataset. Esta etapa es importante porque los modelos pueden fallar o entregar resultados engañosos si existen columnas con espacios, valores vacíos, datos faltantes, tipos incorrectos o variables que filtran información futura.

In [ ]:
# ============================================================
# 5. Limpieza preventiva de nombres y textos
# ============================================================
df = df_original.copy()

# Quitar espacios al inicio y final de los nombres de columnas.
df.columns = df.columns.str.strip()

# Normalizar columnas de texto: quitar espacios, convertir vacíos en nulos.
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('string').str.strip()
    df[col] = df[col].replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

print('Columnas después de normalizar nombres:')
print(df.columns.tolist())

In [ ]:
# ============================================================
# 6. Perfil de calidad de datos
# ============================================================
perfil_calidad = pd.DataFrame({
    'columna': df.columns,
    'tipo': [str(t) for t in df.dtypes],
    'nulos': df.isna().sum().values,
    'porcentaje_nulos': (df.isna().mean() * 100).round(2).values,
    'valores_unicos': df.nunique(dropna=True).values
}).sort_values('porcentaje_nulos', ascending=False)

duplicados = df.duplicated().sum()
print('Duplicados exactos:', duplicados)
perfil_calidad

In [ ]:
# Gráfico de valores faltantes principales
missing_top = perfil_calidad.head(12).sort_values('porcentaje_nulos')
plt.figure(figsize=(8, 5))
plt.barh(missing_top['columna'], missing_top['porcentaje_nulos'])
plt.title('Variables con mayor porcentaje de datos faltantes')
plt.xlabel('% de valores faltantes')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_missing_values.png', bbox_inches='tight')
plt.show()

## Decisiones de limpieza

- No se eliminan todas las filas con nulos, porque eso reduciría mucho el dataset y podría sesgar el análisis.
- Las variables numéricas se imputan con la **mediana**, que es más resistente frente a valores extremos.
- Las variables categóricas se imputan con la **moda**.
- `Date` se transforma a fecha y se crean variables de calendario.
- `RISK_MM` se excluye del modelamiento predictivo, porque representa lluvia futura medida. Usarla sería fuga de información.

In [ ]:
# ============================================================
# 7. Transformación de fecha y variable objetivo
# ============================================================
# Convertir Date a fecha real.
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

def estacion_australia(mes):
    if pd.isna(mes):
        return pd.NA
    mes = int(mes)
    if mes in [12, 1, 2]:
        return 'Verano'
    if mes in [3, 4, 5]:
        return 'Otono'
    if mes in [6, 7, 8]:
        return 'Invierno'
    return 'Primavera'

df['Season'] = df['Month'].apply(estacion_australia).astype('string')

# Eliminar registros sin target, ya que no sirven para entrenar ni evaluar clasificación supervisada.
df_modelo = df.dropna(subset=['RainTomorrow']).copy()
df_modelo['RainTomorrow_bin'] = df_modelo['RainTomorrow'].map({'Yes': 1, 'No': 0}).astype(int)

print('Filas originales:', len(df))
print('Filas para modelamiento:', len(df_modelo))
df_modelo[['Date', 'Year', 'Month', 'Season', 'RainTomorrow', 'RainTomorrow_bin']].head()

In [ ]:
# Distribución de la variable objetivo
objetivo = df_modelo['RainTomorrow_bin'].map({0: 'No', 1: 'Sí'})
porcentaje_objetivo = objetivo.value_counts(normalize=True).sort_index() * 100
print(porcentaje_objetivo.round(2))

plt.figure(figsize=(5, 4))
plt.bar(porcentaje_objetivo.index, porcentaje_objetivo.values)
plt.title('Distribución de RainTomorrow')
plt.ylabel('% de registros')
plt.xlabel('¿Llueve mañana?')
for i, v in enumerate(porcentaje_objetivo.values):
    plt.text(i, v + 1, f'{v:.1f}%', ha='center')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_target_distribution.png', bbox_inches='tight')
plt.show()

# 4. Análisis exploratorio de datos

El análisis exploratorio busca entender qué variables se relacionan con la lluvia del día siguiente y qué patrones aparecen por ubicación, mes y condiciones meteorológicas.

In [ ]:
# ============================================================
# 8. Relación entre lluvia hoy y lluvia mañana
# ============================================================
lluvia_hoy = pd.crosstab(
    df_modelo['RainToday'],
    df_modelo['RainTomorrow_bin'],
    normalize='index'
) * 100
lluvia_hoy.columns = ['No_lueve_manana_%', 'Llueve_manana_%']
lluvia_hoy.round(2)

In [ ]:
# Riesgo por ubicación
riesgo_ubicacion = df_modelo.groupby('Location')['RainTomorrow_bin'].agg(['count', 'mean']).sort_values('mean', ascending=False)
riesgo_ubicacion['riesgo_%'] = (riesgo_ubicacion['mean'] * 100).round(2)
riesgo_ubicacion.head(10)

In [ ]:
plt.figure(figsize=(8, 5))
top_ubicaciones = riesgo_ubicacion['riesgo_%'].head(10).sort_values()
plt.barh(top_ubicaciones.index, top_ubicaciones.values)
plt.title('Ubicaciones con mayor frecuencia de lluvia al día siguiente')
plt.xlabel('% RainTomorrow = Sí')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_top_locations.png', bbox_inches='tight')
plt.show()

In [ ]:
# Riesgo por mes
riesgo_mes = df_modelo.groupby('Month')['RainTomorrow_bin'].mean().mul(100).round(2)
print(riesgo_mes)

plt.figure(figsize=(7, 4))
plt.plot(riesgo_mes.index, riesgo_mes.values, marker='o')
plt.title('Probabilidad de lluvia al día siguiente por mes')
plt.xlabel('Mes')
plt.ylabel('% RainTomorrow = Sí')
plt.xticks(range(1, 13))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_month_risk.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlaciones de variables numéricas con el target
numeric_cols_para_corr = [c for c in df_modelo.select_dtypes(include=np.number).columns if c not in ['RainTomorrow_bin', 'RISK_MM']]
correlaciones = df_modelo[numeric_cols_para_corr + ['RainTomorrow_bin']].corr(numeric_only=True)['RainTomorrow_bin']
correlaciones = correlaciones.drop('RainTomorrow_bin').sort_values(key=abs, ascending=False)
correlaciones.head(12).round(3)

In [ ]:
plt.figure(figsize=(8, 5))
corr_plot = correlaciones.head(10).sort_values()
plt.barh(corr_plot.index, corr_plot.values)
plt.title('Variables numéricas más relacionadas con RainTomorrow')
plt.xlabel('Correlación con RainTomorrow')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_correlations.png', bbox_inches='tight')
plt.show()

## Principales insights exploratorios

- La clase “Sí llueve mañana” es minoritaria, por eso no basta con mirar solo el accuracy.
- La humedad a las 3pm, la nubosidad y la menor cantidad de horas de sol aparecen muy relacionadas con la lluvia posterior.
- Cuando `RainToday = Yes`, la probabilidad de `RainTomorrow = Yes` aumenta de forma importante.
- Existen diferencias claras entre ubicaciones: algunas zonas históricamente tienen mucho más riesgo de lluvia que otras.
- La estacionalidad aporta información útil, por lo que se conserva el mes como variable predictora.

# 5. Preparación final para modelos predictivos

En esta etapa se separan variables predictoras y objetivo, se elimina fuga de información y se construye un pipeline para que la preparación sea reproducible.

In [ ]:
# ============================================================
# 9. Selección de variables y prevención de fuga de información
# ============================================================
columnas_no_predictoras = ['RainTomorrow', 'RainTomorrow_bin', 'RISK_MM', 'Date']
X = df_modelo.drop(columns=[c for c in columnas_no_predictoras if c in df_modelo.columns])
y = df_modelo['RainTomorrow_bin']

# Convertir pd.NA a np.nan en texto para compatibilidad con scikit-learn.
for col in X.select_dtypes(include=['object', 'string']).columns:
    X[col] = X[col].astype('object').where(X[col].notna(), np.nan)

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

print('Variables numéricas:', len(num_cols), num_cols)
print('Variables categóricas:', len(cat_cols), cat_cols)
print('Total variables predictoras antes de One-Hot:', X.shape[1])

In [ ]:
# División de entrenamiento y prueba con estratificación.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Proporción clase positiva en train:', round(y_train.mean() * 100, 2), '%')
print('Proporción clase positiva en test:', round(y_test.mean() * 100, 2), '%')

In [ ]:
# Pipeline de preprocesamiento
numeric_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

preprocess = ColumnTransformer(transformers=[
    ('num', numeric_pipe, num_cols),
    ('cat', categorical_pipe, cat_cols)
])

print('Pipeline de preparación definido.')

# 6. Aplicación de técnicas de minería de datos

Se aplican modelos de clasificación para predecir lluvia al día siguiente. Se incluye un baseline de mayoría para demostrar que un accuracy alto puede ser engañoso cuando los datos están desbalanceados.

In [ ]:
# ============================================================
# 10. Entrenamiento de modelos
# ============================================================
# Para que el notebook sea ejecutable de forma razonable en Colab, se permite entrenar los modelos principales con una muestra estratificada.
# El análisis exploratorio y las evaluaciones se mantienen sobre el dataset completo de prueba.
MODEL_SAMPLE = 50000

def sample_train(X_train, y_train, n=MODEL_SAMPLE):
    if len(X_train) <= n:
        return X_train, y_train
    X_fit, _, y_fit, _ = train_test_split(X_train, y_train, train_size=n, stratify=y_train, random_state=42)
    return X_fit, y_fit

X_fit, y_fit = sample_train(X_train, y_train)
print('Filas usadas para entrenar modelos principales:', len(X_fit))

modelos = {
    'Baseline_mayoria': Pipeline(steps=[
        ('prep', preprocess),
        ('model', DummyClassifier(strategy='most_frequent'))
    ]),
    'Regresion_Logistica': Pipeline(steps=[
        ('prep', preprocess),
        ('model', LogisticRegression(max_iter=400, class_weight='balanced', solver='liblinear', random_state=42))
    ]),
    'Arbol_Decision': Pipeline(steps=[
        ('prep', preprocess),
        ('model', DecisionTreeClassifier(max_depth=12, min_samples_leaf=40, class_weight='balanced', random_state=42))
    ])
}

resultados = []
predicciones = {}

for nombre, pipe in modelos.items():
    print('Entrenando:', nombre)
    if nombre == 'Baseline_mayoria':
        pipe.fit(X_train, y_train)
    else:
        pipe.fit(X_fit, y_fit)
    y_pred = pipe.predict(X_test)
    if hasattr(pipe[-1], 'predict_proba'):
        y_prob = pipe.predict_proba(X_test)[:, 1]
    else:
        y_prob = None
    cm = confusion_matrix(y_test, y_pred)
    resultados.append({
        'modelo': nombre,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan,
        'tn': cm[0,0], 'fp': cm[0,1], 'fn': cm[1,0], 'tp': cm[1,1]
    })
    predicciones[nombre] = {'pred': y_pred, 'prob': y_prob, 'pipeline': pipe}

metricas = pd.DataFrame(resultados).sort_values('f1', ascending=False)
metricas.to_csv(OUTPUT_DIR / 'metricas_modelos.csv', index=False)
metricas.round(3)

In [ ]:
plt.figure(figsize=(8, 4.5))
metricas.set_index('modelo')[['precision', 'recall', 'f1', 'roc_auc']].plot(kind='bar')
plt.title('Comparación de modelos predictivos')
plt.ylim(0, 1)
plt.ylabel('Métrica')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_model_metrics.png', bbox_inches='tight')
plt.show()

In [ ]:
# Modelo seleccionado según F1
modelo_ganador = metricas.iloc[0]['modelo']
best = predicciones[modelo_ganador]
print('Modelo seleccionado:', modelo_ganador)
print(classification_report(y_test, best['pred'], target_names=['No llueve', 'Llueve']))

cm = confusion_matrix(y_test, best['pred'])
plt.figure(figsize=(4.5, 4))
plt.imshow(cm)
plt.title(f'Matriz de confusión - {modelo_ganador}')
plt.xticks([0, 1], ['Pred No', 'Pred Sí'])
plt.yticks([0, 1], ['Real No', 'Real Sí'])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=12)
plt.colorbar(fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_confusion_matrix.png', bbox_inches='tight')
plt.show()

## Evaluación del modelo

La evaluación se realiza con varias métricas:

- **Accuracy:** proporción general de aciertos.
- **Precision:** de los casos predichos como lluvia, cuántos realmente fueron lluvia.
- **Recall:** de los días que realmente tuvieron lluvia, cuántos fueron detectados por el modelo.
- **F1-score:** equilibrio entre precision y recall.
- **ROC-AUC:** capacidad del modelo para separar las clases.

En este problema el recall es importante porque no detectar un día lluvioso puede afectar decisiones operativas. Sin embargo, la precision también debe controlarse para no generar demasiadas alertas innecesarias.

In [ ]:
# Ajuste opcional del umbral para el modelo ganador
if best['prob'] is not None:
    thresholds = np.linspace(0.05, 0.95, 181)
    umbrales = []
    for t in thresholds:
        pred_t = (best['prob'] >= t).astype(int)
        umbrales.append({
            'threshold': t,
            'precision': precision_score(y_test, pred_t, zero_division=0),
            'recall': recall_score(y_test, pred_t, zero_division=0),
            'f1': f1_score(y_test, pred_t, zero_division=0)
        })
    umbrales_df = pd.DataFrame(umbrales)
    mejor_umbral = umbrales_df.loc[umbrales_df['f1'].idxmax()]
    umbrales_df.to_csv(OUTPUT_DIR / 'umbral_f1_modelo_ganador.csv', index=False)
    display(mejor_umbral)
else:
    print('El modelo ganador no entrega probabilidades.')

# 7. Descubrimiento de patrones con clustering

Para complementar la predicción se aplica K-Means sobre variables meteorológicas numéricas. La idea es descubrir grupos de días con comportamiento similar sin usar `RainTomorrow` como variable de entrada. Después se cruza cada grupo con el resultado real de lluvia para interpretar su riesgo.

In [ ]:
# ============================================================
# 11. Clustering meteorológico
# ============================================================
cluster_vars = [
    'MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed',
    'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm',
    'Temp9am', 'Temp3pm', 'Cloud9am', 'Cloud3pm', 'Evaporation', 'Sunshine'
]
cluster_vars = [c for c in cluster_vars if c in df_modelo.columns]

cluster_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

X_cluster = cluster_pipe.fit_transform(df_modelo[cluster_vars])

# Se usa MiniBatchKMeans por eficiencia. k=4 permite una interpretación clara de perfiles.
k = 4
rng = np.random.default_rng(42)
idx = rng.choice(len(X_cluster), min(20000, len(X_cluster)), replace=False)

kmeans = MiniBatchKMeans(n_clusters=k, n_init=3, batch_size=2048, max_iter=50, random_state=42)
kmeans.fit(X_cluster[idx])
clusters = kmeans.predict(X_cluster)

df_clusters = df_modelo[['Location', 'Month', 'RainToday', 'RainTomorrow_bin'] + cluster_vars].copy()
df_clusters['Cluster'] = clusters

perfil_clusters = df_clusters.groupby('Cluster').agg(
    dias=('Cluster', 'count'),
    prob_lluvia_manana=('RainTomorrow_bin', 'mean'),
    humedad_3pm=('Humidity3pm', 'mean'),
    lluvia_mm=('Rainfall', 'mean'),
    presion_3pm=('Pressure3pm', 'mean'),
    temp_3pm=('Temp3pm', 'mean'),
    nubosidad_3pm=('Cloud3pm', 'mean')
).round(3)
perfil_clusters['prob_lluvia_manana_%'] = (perfil_clusters['prob_lluvia_manana'] * 100).round(2)
perfil_clusters.to_csv(OUTPUT_DIR / 'perfil_clusters.csv')
perfil_clusters

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(perfil_clusters.index.astype(str), perfil_clusters['prob_lluvia_manana_%'])
plt.title('Riesgo de lluvia por cluster meteorológico')
plt.xlabel('Cluster')
plt.ylabel('% RainTomorrow = Sí')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_clusters.png', bbox_inches='tight')
plt.show()

# Visualización 2D con PCA para una muestra
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_cluster[idx][:5000])
labels_sample = kmeans.predict(X_cluster[idx][:5000])

plt.figure(figsize=(6, 4.5))
plt.scatter(coords[:, 0], coords[:, 1], c=labels_sample, s=3, alpha=0.5)
plt.title('Visualización PCA de clusters (muestra)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_pca_clusters.png', bbox_inches='tight')
plt.show()

## Insights de patrones

- El cluster con mayor riesgo reúne días con más humedad en la tarde, más lluvia registrada, menor presión relativa y mayor nubosidad.
- Los clusters más secos tienen baja humedad y menor lluvia observada, por lo que su probabilidad de lluvia al día siguiente baja de forma importante.
- Esta segmentación permite diferenciar alertas: no todos los días ni todas las ubicaciones necesitan la misma respuesta operativa.
- El análisis por ubicación muestra que el riesgo histórico no es homogéneo; algunas zonas requieren mayor vigilancia que otras.

# 8. Documentación del proceso y archivos de salida

Se guardan archivos de respaldo para que el trabajo sea trazable: perfil de calidad, métricas, riesgos por ubicación, riesgos por mes, clusters, dataset preparado y modelo seleccionado.

In [ ]:
# ============================================================
# 12. Guardado de salidas
# ============================================================
perfil_calidad.to_csv(OUTPUT_DIR / 'perfil_calidad_datos.csv', index=False)
riesgo_ubicacion.to_csv(OUTPUT_DIR / 'riesgo_lluvia_por_ubicacion.csv')
riesgo_mes.to_csv(OUTPUT_DIR / 'riesgo_lluvia_por_mes.csv')
correlaciones.to_csv(OUTPUT_DIR / 'correlaciones_con_target.csv')
lluvia_hoy.to_csv(OUTPUT_DIR / 'lluvia_manana_por_lluvia_hoy.csv')

# Dataset preparado base: se guarda sin la columna original de etiqueta textual para evitar duplicidad.
df_preparado = df_modelo.drop(columns=['RainTomorrow'], errors='ignore')
df_preparado.to_csv(OUTPUT_DIR / 'weatherAUS_preparado_base.csv', index=False)

# Guardar modelo ganador.
joblib.dump(best['pipeline'], OUTPUT_DIR / 'modelo_lluvia_weatherAUS.joblib')

print('Archivos generados:')
for archivo in sorted(OUTPUT_DIR.glob('*')):
    print('-', archivo.name)

# 9. Conclusiones

El proyecto demuestra que `weatherAUS.csv` permite construir una solución predictiva útil para anticipar lluvia al día siguiente. El comportamiento de la lluvia no depende de una sola variable, pero sí aparecen señales claras: humedad en la tarde, nubosidad, lluvia del día actual, presión y horas de sol.

El modelo seleccionado debe evaluarse considerando el contexto. En decisiones preventivas, puede ser preferible detectar más días lluviosos aunque existan algunas alertas adicionales. Por eso se revisa F1, recall y ROC-AUC, y no solo accuracy.

El clustering complementa la predicción porque entrega perfiles interpretables de días meteorológicos. Esto ayuda a transformar el análisis en acciones: priorizar ubicaciones, ajustar turnos, preparar recursos y definir niveles de alerta.

# 10. Recomendaciones y automatización

## Acciones recomendadas

1. Usar el modelo como apoyo preventivo para priorizar días con riesgo de lluvia.
2. Implementar un tablero por ubicación, fecha, probabilidad de lluvia y nivel de riesgo.
3. Mantener controles de calidad para datos faltantes y categorías nuevas.
4. No usar `RISK_MM` como predictor en producción, porque filtra información futura.
5. Monitorear el rendimiento del modelo periódicamente con F1, recall y ROC-AUC.

## Siguientes pasos

- Incorporar más fuentes, como pronósticos oficiales y coordenadas geográficas.
- Evaluar modelos adicionales con validación temporal.
- Ajustar umbrales según el costo real de una falsa alarma versus no anticipar lluvia.
- Automatizar el flujo completo: carga diaria, limpieza, predicción, almacenamiento y alerta.

# 11. Reflexión final

El trabajo permite recorrer un ciclo completo de minería de datos, desde la comprensión del problema hasta un modelo evaluado y una propuesta de automatización. La parte más importante no fue solo entrenar modelos, sino tomar decisiones correctas sobre la calidad de datos, la fuga de información, el desbalance de clases y la interpretación de resultados.

Este enfoque puede adaptarse a otros problemas de negocio donde existan datos históricos y se necesite anticipar eventos futuros.

# 12. Checklist contra pauta de evaluación

- **Introducción:** objetivo, contexto y aporte de minería de datos.
- **Metodología:** CRISP-DM, análisis del problema, fuentes y técnicas justificadas.
- **Preparación de datos:** obtención, limpieza, imputación, transformación, EDA y control de fuga de información.
- **Aplicación de técnicas:** clasificación y clustering, con explicación de implementación.
- **Modelos predictivos:** baseline, regresión logística, árbol de decisión, métricas y matriz de confusión.
- **Descubrimiento de patrones:** análisis por variables, ubicación, mes y clusters.
- **Documentación:** proceso completo, desafíos, salidas generadas e insights.
- **Automatización:** propuesta de pipeline diario y monitoreo.
- **Recomendaciones:** acciones concretas y próximos pasos.
- **Referencias:** metodología, herramientas y dataset utilizado.

# 13. Referencias

- Chapman, P. et al. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*.
- Kuhn, M. & Johnson, K. (2013). *Applied Predictive Modeling*. Springer.
- Pandas Development Team. Documentación oficial de pandas.
- Scikit-learn Developers. Documentación oficial de scikit-learn.
- Matplotlib Development Team. Documentación oficial de Matplotlib.
- Dataset utilizado: `weatherAUS.csv`, archivo entregado para la evaluación BIY7121.